In [ ]:
import sys
import os
import torch
import torch.nn as nn
import torch.optim as optim
from google.colab import drive

# ---------------------------------------------------------
# 1. SETUP AMBIENTE E PERCORSI (Drive + GitHub)
# ---------------------------------------------------------
# Monta Google Drive per accedere a Dati e salvare i Pesi
drive.mount('/content/drive')

# Definisci i percorsi su Drive (modificali in base alle tue cartelle)
DRIVE_DATA_DIR = '/content/drive/MyDrive/WiFi-HAR/doppler_traces'
DRIVE_WEIGHTS_DIR = '/content/drive/MyDrive/WiFi-HAR/weights'

# Crea la cartella dei pesi se non esiste ancora
os.makedirs(DRIVE_WEIGHTS_DIR, exist_ok=True)

# Clona la TUA repository GitHub per scaricare i moduli .py
# Sostituisci l'URL con quello della repo che condividi col tuo collega
REPO_URL = 'https:/https://github.com/teoosera/WiFi-HAR.git'
!rm -rf /content/repo_code 
!git clone {REPO_URL} /content/repo_code

# Aggiunge la cartella al path per permettere l'importazione
sys.path.append('/content/repo_code')

# ---------------------------------------------------------
# 2. IMPORTAZIONE DAI TUOI MODULI
# ---------------------------------------------------------
from dataset import get_dataloaders
from models import CustomSimpleCNN, CustomResNet, CNN_GRU
from train import train_one_epoch
from evaluate import evaluate_decision_fusion

# ---------------------------------------------------------
# 3. INIZIALIZZAZIONE DEL MODELLO
# ---------------------------------------------------------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Utilizzando device: {device}")

# SCEGLI QUI QUALE DEI 3 MODELLI TESTARE:
# model = CustomSimpleCNN(num_classes=5).to(device)
model = CustomResNet(num_classes=5).to(device)
# model = CNN_GRU(num_classes=5).to(device)

optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

# ---------------------------------------------------------
# 4. CARICAMENTO DATI E TRAINING LOOP
# ---------------------------------------------------------
train_loader, test_loader = get_dataloaders(DRIVE_DATA_DIR, batch_size=32)

num_epochs = 10
for epoch in range(num_epochs):
    print(f"\n--- Epoca {epoch+1}/{num_epochs} ---")
    
    # Fase di addestramento
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    
    # Fase di Test (utilizza la Decision Fusion sulle 4 antenne)
    evaluate_decision_fusion(model, test_loader, device)

# ---------------------------------------------------------
# 5. SALVATAGGIO DEI PESI SU GOOGLE DRIVE
# ---------------------------------------------------------
# Il nome del file può essere cambiato in base al modello in uso
save_path = f"{DRIVE_WEIGHTS_DIR}/custom_resnet_finale.pth"
torch.save(model.state_dict(), save_path)
print(f"\nAddestramento concluso! Pesi salvati in modo permanente su Drive in: {save_path}")